# WeatherAPI → Fabric EventStream (PySpark + HTTP)
Fetches **full weather data** (current + air quality + forecast + alerts)
from WeatherAPI.com and pushes to **Weather-EventStream** every **30 seconds** via HTTP POST.

**Flow:** WeatherAPI.com → PySpark → HTTP POST → Weather-EventStream

In [1]:
# ─── 1. Configuration ────────────────────────────────────────────────────────

# --- WeatherAPI.com ---
WEATHER_API_KEY = "22589cd3dc2f47538a574004261704"
CITIES          = ["Noida"]
WEATHER_API_URL = "https://api.weatherapi.com/v1/forecast.json"

# --- Fabric EventStream Custom Endpoint ---
NAMESPACE     = "esehpn8h6syyxd7q3qi1hy"
EVENTHUB_NAME = "es_8c785007-dc2f-4513-ab81-ab2f23850c52"
SAS_KEY_NAME  = "key_5b40ed95-d5c8-4d85-9aaf-68c149d94bd3"
SAS_KEY_VALUE = "a5OG/uNJWI2JLxis7Z6Ia9lgCdtPH8ShD+AEhHyVL38="

# --- Polling ---
POLL_INTERVAL_SECONDS = 30   # fetch every 30 seconds
MAX_CYCLES            = None  # None = run indefinitely until manually stopped
                              # Set to a number e.g. 10 to run for finite cycles

StatementMeta(, 3478a0a7-4780-46f6-a6a3-0f5f2b113e3f, 3, Finished, Available, Finished, False)

In [2]:
# ─── 2. Imports (all built-in, zero installs needed) ──────────────────────
import requests
import json
import time
import hmac
import hashlib
import base64
import urllib.parse
from datetime import datetime, timezone, timedelta

StatementMeta(, 3478a0a7-4780-46f6-a6a3-0f5f2b113e3f, 4, Finished, Available, Finished, False)

In [3]:
# ─── 3. SAS Token Generator (no Azure SDK) ───────────────────────────────
def generate_sas_token(namespace, eventhub, key_name, key_value, expiry_hours=1):
    uri         = f"https://{namespace}.servicebus.windows.net/{eventhub}"
    encoded_uri = urllib.parse.quote_plus(uri)
    expiry      = int((datetime.now(timezone.utc) + timedelta(hours=expiry_hours)).timestamp())
    to_sign     = f"{encoded_uri}\n{expiry}"
    signature   = base64.b64encode(
        hmac.new(key_value.encode(), to_sign.encode(), hashlib.sha256).digest()
    ).decode()
    return (
        f"SharedAccessSignature sr={encoded_uri}"
        f"&sig={urllib.parse.quote_plus(signature)}"
        f"&se={expiry}"
        f"&skn={key_name}"
    )

EVENTSTREAM_URL = f"https://{NAMESPACE}.servicebus.windows.net/{EVENTHUB_NAME}/messages"
print(f"EventStream URL : {EVENTSTREAM_URL}")
print(f"Poll interval   : {POLL_INTERVAL_SECONDS}s")
print(f"Max cycles      : {'infinite' if MAX_CYCLES is None else MAX_CYCLES}")

StatementMeta(, 3478a0a7-4780-46f6-a6a3-0f5f2b113e3f, 5, Finished, Available, Finished, False)

EventStream URL : https://esehpn8h6syyxd7q3qi1hy.servicebus.windows.net/es_8c785007-dc2f-4513-ab81-ab2f23850c52/messages
Poll interval   : 30s
Max cycles      : infinite


In [4]:
# ─── 4. Fetch ALL weather fields from WeatherAPI.com ───────────────────────
def fetch_weather(city):
    try:
        resp = requests.get(
            WEATHER_API_URL,
            params={"key": WEATHER_API_KEY, "q": city,
                    "days": 3, "aqi": "yes", "alerts": "yes"},
            timeout=10
        )
        resp.raise_for_status()
        d   = resp.json()
        loc = d["location"]
        cur = d["current"]
        aqi = cur.get("air_quality", {})
        return {
            # Location
            "name":           loc["name"],
            "region":         loc["region"],
            "country":        loc["country"],
            "lat":            loc["lat"],
            "lon":            loc["lon"],
            "localtime":      loc["localtime"],
            # Current
            "temp_c":         cur["temp_c"],
            "is_day":         cur["is_day"],
            "condition_text": cur["condition"]["text"],
            "condition_icon": cur["condition"]["icon"],
            "wind_kph":       cur["wind_kph"],
            "wind_degree":    cur["wind_degree"],
            "wind_dir":       cur["wind_dir"],
            "pressure_mb":    cur["pressure_mb"],
            "precip_mm":      cur["precip_mm"],
            "humidity":       cur["humidity"],
            "cloud":          cur["cloud"],
            "feelslike_c":    cur["feelslike_c"],
            "uv":             cur["uv"],
            # Air Quality
            "air_quality": {
                "co":             aqi.get("co"),
                "no2":            aqi.get("no2"),
                "o3":             aqi.get("o3"),
                "so2":            aqi.get("so2"),
                "pm2_5":          aqi.get("pm2_5"),
                "pm10":           aqi.get("pm10"),
                "us_epa_index":   aqi.get("us-epa-index"),
                "gb_defra_index": aqi.get("gb-defra-index")
            },
            # 3-day Forecast
            "forecast": [
                {
                    "date":      day["date"],
                    "maxtemp_c": day["day"]["maxtemp_c"],
                    "mintemp_c": day["day"]["mintemp_c"],
                    "condition": day["day"]["condition"]["text"]
                }
                for day in d.get("forecast", {}).get("forecastday", [])
            ],
            # Alerts
            "alerts": [
                {
                    "headline":  a.get("headline"),
                    "severity":  a.get("severity"),
                    "event":     a.get("event"),
                    "effective": a.get("effective"),
                    "expires":   a.get("expires"),
                    "desc":      a.get("desc")
                }
                for a in d.get("alerts", {}).get("alert", [])
            ],
            # Ingestion metadata
            "ingested_at_utc": datetime.now(timezone.utc).isoformat()
        }
    except Exception as e:
        print(f"  [ERROR] {city}: {e}")
        return None

StatementMeta(, 3478a0a7-4780-46f6-a6a3-0f5f2b113e3f, 6, Finished, Available, Finished, False)

In [5]:
# ─── 5. Send events to EventStream via HTTP POST ───────────────────────────
def send_to_eventstream(records):
    sas_token = generate_sas_token(NAMESPACE, EVENTHUB_NAME, SAS_KEY_NAME, SAS_KEY_VALUE)
    headers   = {"Authorization": sas_token, "Content-Type": "application/json"}
    ok = 0
    for record in records:
        resp = requests.post(EVENTSTREAM_URL, headers=headers,
                             data=json.dumps(record), timeout=10)
        if resp.status_code == 201:
            ok += 1
        else:
            print(f"  [WARN] {record.get('name')}: HTTP {resp.status_code} — {resp.text}")
    print(f"  [✓] Sent {ok}/{len(records)} events at {datetime.now(timezone.utc).isoformat()}")

StatementMeta(, 3478a0a7-4780-46f6-a6a3-0f5f2b113e3f, 7, Finished, Available, Finished, False)

In [ ]:
# ─── 6. Main loop — runs every 30 seconds ──────────────────────────────────
# Runs indefinitely when MAX_CYCLES = None
# Interrupt the kernel manually to stop
cycle = 0
while True:
    cycle += 1
    label = f"Cycle {cycle}" if MAX_CYCLES is None else f"Cycle {cycle}/{MAX_CYCLES}"
    print(f"\n{'='*65}")
    print(f" {label} — {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print(f"{'='*65}")

    # Step 1: Fetch weather for all cities
    records = []
    for city in CITIES:
        data = fetch_weather(city)
        if data:
            aq = data["air_quality"]
            print(f"  ✓ {data['name']}, {data['region']} | "
                  f"{data['temp_c']}°C | {data['condition_text']} | "
                  f"AQI(EPA):{aq['us_epa_index']} | PM2.5:{aq['pm2_5']} | "
                  f"Alerts:{len(data['alerts'])}")
            records.append(data)

    # Step 2: Push to EventStream
    if records:
        send_to_eventstream(records)
    else:
        print("  [WARN] No records fetched this cycle.")

    # Step 3: Check if max cycles reached
    if MAX_CYCLES is not None and cycle >= MAX_CYCLES:
        print(f"\n✓ Done. {MAX_CYCLES} cycle(s) completed.")
        break

    # Step 4: Wait 30 seconds before next fetch
    print(f"  ⏳ Next fetch in {POLL_INTERVAL_SECONDS}s... (interrupt kernel to stop)")
    time.sleep(POLL_INTERVAL_SECONDS)

  ✓ Noida, Uttar Pradesh | 30.4°C | Sunny | AQI(EPA):5 | PM2.5:218.75 | Alerts:0
  [✓] Sent 1/1 events at 2026-05-07T11:57:57.801573+00:00
  ⏳ Next fetch in 30s... (interrupt kernel to stop)

 Cycle 11887 — 2026-05-07 11:58:27 UTC
  ✓ Noida, Uttar Pradesh | 30.4°C | Sunny | AQI(EPA):5 | PM2.5:218.75 | Alerts:0
  [✓] Sent 1/1 events at 2026-05-07T11:58:28.161886+00:00
  ⏳ Next fetch in 30s... (interrupt kernel to stop)

 Cycle 11888 — 2026-05-07 11:58:58 UTC
  ✓ Noida, Uttar Pradesh | 30.4°C | Sunny | AQI(EPA):5 | PM2.5:218.75 | Alerts:0
  [✓] Sent 1/1 events at 2026-05-07T11:58:58.524106+00:00
  ⏳ Next fetch in 30s... (interrupt kernel to stop)

 Cycle 11889 — 2026-05-07 11:59:28 UTC
  ✓ Noida, Uttar Pradesh | 30.4°C | Sunny | AQI(EPA):5 | PM2.5:218.75 | Alerts:0
  [✓] Sent 1/1 events at 2026-05-07T11:59:28.896108+00:00
  ⏳ Next fetch in 30s... (interrupt kernel to stop)

 Cycle 11890 — 2026-05-07 11:59:58 UTC
  ✓ Noida, Uttar Pradesh | 30.4°C | Sunny | AQI(EPA):5 | PM2.5:218.75 | Aler